<a href="https://www.kaggle.com/code/adrinorosario/undiagnosed-gemma4-medicaldocumentparser?scriptVersionId=323958547" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Document Parser

This is where Agent 1 will be built. The role of Agent 1 is to be a document parser; take any raw medical document -- PDF or Image -- and extract a structured clinical profile as output.

This is the approach that needs to be followed:
1. Read the respective document. They can be of the following types:
    * A digital text document where everything is in a text format
    * A PDF/Image that contains pixel and related data
    * A mix of both
      
Each of them will follow different approaches

In [1]:
# dependencies required to be satisfied for the core dependencies installed in the following cell
!pip install "opentelemetry-exporter-otlp-proto-grpc==1.38.0."
!pip install "opentelemetry-api<1.40.0" "opentelemetry-sdk<1.39.0" "opentelemetry-proto==1.38.0" "opentelemetry-exporter-otlp-proto-common==1.38.0" google-cloud-bigquery-storage

!pip install -q pymupdf sentence-transformers pillow bitsandbytes
!pip install -q bitsandbytes>=0.45.0 --upgrade

# pull transformers directly from GitHub repository since Gemma's architecture is fairly new, and the current library does not recognise it
!pip install -U git+https://github.com/huggingface/transformers.git

"""
- transformers from HuggingFace for model definition, inference pipelines, training, and output generation
- accelerate enables PyTorch code to run across any distributed configuration
- pymupdf to extract data, analysis, conversion, and manipulating pdfs
- sentence-transformers for computing embeddings from text
- pillow for image manipulation across various formats
- bitsandbytes for quantization
""" 

import accelerate
import transformers
print("accelerate:", accelerate.__version__)   # should be 1.x+
print("transformers:", transformers.__version__)

ERROR: Invalid requirement: 'opentelemetry-exporter-otlp-proto-grpc==1.38.0.': Expected end or semicolon (after version specifier)
    opentelemetry-exporter-otlp-proto-grpc==1.38.0.
                                          ~~~~~~~~^
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.7/306.7 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.6 MB/s eta 0:00:00
  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-ni_9dftm
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-ni_9dftm
  Resolved https://github.com/huggingface/transformers.git to commit 9e9ed4e6e64ba342cefcab2c2b927e3b8558caa1
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 671.5/671.5 kB 31

## Strategies for architecting the document parsing pipeline

Determining a singular pipeline that works for all documents is not computationally possible with the number of tradeoffs it carries. Therefore, the pipeline will be broken down into modular phases or steps that gradually break down the complexities.

* Phase 1: This is the phase where you determine the nature of the document you are dealing with.
* Phase 2: This phase focuses on converting the raw data and signals into clean and structured strings that maintain context and spatial truth.
* Phase 3: Text vs Vision Strategy - this is more about whether a more vision approach is better than a fully-textual approach. More on this later.
* Phase 4: The outputs of the different modules are synthesized into one single LLM-optimized representation or, in this instance, the Gemma-4 format.

### Phase 1: Classification

This stage is crucial as it will determine the quality of the data we ultimately feed into the model. In essence, we can perform certain steps to ensure computation is not wasted, and maximise accuracy:
* Analyse the PDF object tree and compare the count of XObjects (reusable graphic objects in a PDF, such as images, forms, etc.) vs Font objects. A higher count of the former indicates a scanned document.
* Check for the hidden text layer. If it exists, perform a garbage check comparing the number of white spaces to characters, non-dictionary word frequency, etc.
* Branching logic:
    1. If it is a digital native document, perform a direct stream operation and extract the text
    2. If it is a scanned document, convert the full page with the right rendering parameters for OCR
    3. If it is a hybrid (text + images scanned), extract text and separate images for vision processing 

In [2]:
!pip install --upgrade pymupdf 

### Imports

| Module | Role |
|---|---|
| `fitz` (PyMuPDF) | PDF parsing, page rendering, font/XObject analysis |
| `transformers` | Gemma 4 model and processor loading |
| `torch` | GPU inference and memory management |
| `PIL.Image` | Raster image loading and format conversion |
| `base64` | Encoding pages for the vision model |
| `logging` | Structured logging throughout the pipeline |
| `json` | Parsing and validating model JSON output |
| `dataclasses` | Typed containers for processed document data |

In [3]:
from dataclasses import dataclass
from typing import Literal
import fitz as fz
from pathlib import Path
import base64
import logging
import io
from PIL import Image

# for the gemma model call
from transformers import AutoProcessor, Gemma4ForConditionalGeneration
import torch
import json
import gc

### Logger Configuration

A module-level logger replaces raw `print` statements throughout the pipeline. This gives level-controlled, structured output and makes it straightforward to route logs to external observability tools (OpenTelemetry, Cloud Logging) when moving to production.

In [4]:
# Configure module-level logger (adjust level as needed)
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

### Custom Exception Classes

Three exceptions provide precise error signals at each validation stage. Each carries a numeric error code for unambiguous downstream filtering.

| Exception | Code | Raised when |
|---|---|---|
| `IncompatibleFileFormatException` | 700 | Extension not in the supported set |
| `EmptyFileExtensionException` | 700.2 | File has no extension at all |
| `ImageEncodingException` | 701 | Base64 encoding of image bytes fails |

In [5]:
"""
Custom error codes used for custom built exceptions

IncompatibleFileFormatException: 700
EmptyFileExtensionException: 700.2
ImageEncodingFunctionException: 701
ImageBoundingBoxIdentificationException: 707
ImageBase64EncodingException: 714
"""

class ImageEncodingException(Exception):
    """Custom exception to handle image encoding exceptions during document parsing

    Args:
        Exception: Inheriting from the base Exception class
    """

    def __init__(self, message, error_code=701):
        super().__init__(message)
        self.message = message
        self.error_code = error_code
    
    def __str__(self):
        return f"{self.message} (Error code: {self.error_code})"

class IncompatibleFileFormatException(Exception):
    """Handles incompatible file format errors arising from pre-processing validation

    Args:
        Exception: Inheriting from the base Exception class
    """

    def __init__(self, message, error_code=700):
        super().__init__(message)
        self.message = message
        self.error_code = error_code
    
    def __str__(self):
        return f"{self.message} (Error code: {self.error_code})"

class EmptyFileExtensionException(Exception):
    """Handles files that do not have a suffix(extension), arising from pre-processing validation

    Args:
        Exception: Inheriting from the base Exception class
    """

    def __init__(self, message, error_code=700.2):
        super().__init__(message)
        self.message = message
        self.error_code = error_code
    
    def __str__(self):
        return f"{self.message} (Error code: {self.error_code})"

### Supported Format Registries

Two sets define accepted input types:

- **`raster_formats`** — image formats a user might upload (JPEG variants, PNG, TIFF, HEIC, RAW camera formats, etc.)
- **`file_formats`** — document types processed via PyMuPDF (`.pdf`, `.txt`)

Adding a new format is a one-line change to the appropriate set.

In [6]:
# when a user upload an image file, it most probably will be in one of these formats
raster_formats = {
    ".jpg", ".jpeg", ".jpe", ".jif", ".jfif", 
    ".png", 
    ".webp", 
    ".tif", ".tiff", 
    ".heic", ".heif", 
    ".bmp", 
    ".raw", ".cr2", ".nef", ".arw"
}

# for any kind of documents that can be uploaded. for now only these formats are supported; later more can be accommodated
file_formats = {
    ".pdf", ".txt"
}

### Intermediate Representation — Data Classes

Three dataclasses act as typed containers flowing between pipeline stages:

- **`ProcessedPage`** — one page's content after extraction, tagged with `"text"` or `"vision"` to indicate how it was extracted.
- **`ProcessedImage`** — a standalone raster file encoded as base64; always uses `"vision"`.
- **`ProcessedDocument`** — aggregates all `ProcessedPage` objects for a multi-page PDF along with file-level metadata.

These are passed directly to `extract_clinical_signals()`, which uses `match` on the type to route correctly.

In [7]:
# data classes to store the contents of the processed document
@dataclass
class ProcessedPage:
    page_number: int
    extraction_method: Literal["text", "vision"]
    raw_content: str | None # populated by the text extraction and stored as a string
    image_b64_encoded: str | None # populated by the vision extraction which is b64 encoded

@dataclass # similar data class but for raster files
class ProcessedImage:
    file_path: str
    extraction_method: Literal["vision"]
    image_b64_encoded: str | None
@dataclass
class ProcessedDocument:
    file_path: str
    total_pages: int
    pages: list[ProcessedPage]

## Phase 1 — File Validation & Classification

Four functions form the validation layer. They run before any extraction and determine which downstream path a file takes.

### `get_file_extension`

Extracts and lowercases the file suffix. Raises `EmptyFileExtensionException` if no extension is found — preventing silent failures where an extensionless binary might be misrouted.

In [8]:
def get_file_extension(file_path: Path) -> str:
    """Extracts file suffix (extension) and returns it as a string. Raises EmptyFileExtensionException if no extension found

    Args:
        file_path (Path): A Path object to the file

    Returns:
        str: Returns the extension (e.g. .png) as a string
    """

    extension = file_path.suffix.lower()
    # [CHANGED] Replaced print with logger for better observability
    logger.info(f"Extension of {file_path} is {extension}")

    if extension == '' or not extension:
        raise EmptyFileExtensionException(message=f"{file_path} does not have an extension; cannot proceed with further validation\n")
    
    return extension

### `normalize_file_path`

Confirms the path exists and points to a file (not a directory), then delegates to `get_file_extension`. Returns a `(Path, extension)` tuple. Any extension error is re-raised as `IncompatibleFileFormatException` to keep the exception surface consistent for callers.

In [9]:
def normalize_file_path(file_path: str) -> tuple[Path, str]:
    """Normalizes the provided path after validation and returns a Path object for further I/O operations

    Args:
        file_path (str): The file path to the document that needs to be processed

    Returns:
        tuple[Path, str]: A Path object of the file location and the extension of the file
    """
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"Cannot find the path specified: {file_path}")
    
    if not file_path.is_file():
        raise IsADirectoryError(f"Expected a file, but found a directory: {file_path}")
    

    try:
        extension = get_file_extension(file_path=file_path)
        return (file_path, extension)
    except Exception as e:
        # Re-raise to let caller handle; preserves context for debugging
        raise IncompatibleFileFormatException(
            message=f"Failed to normalize file path {file_path}: {e}"
        ) from e

### `classify_file_type`

Maps a file extension to one of two processing categories: `"raster"` or `"document"`. Raises `IncompatibleFileFormatException` for anything outside the registered sets.

In [10]:
def classify_file_type(extension: str) -> Literal["raster", "document"]:
    """Classifies the file for further processing and extraction
    """
    if extension in raster_formats:
        return "raster"
    elif extension in file_formats:
        return "document"
    else:
        raise IncompatibleFileFormatException(message=f"{extension} is not a supported file type\n")

### `document_validator`

The public entry point for Phase 1. Chains `normalize_file_path` → `classify_file_type` and returns a `(file_path, is_valid, file_type)` tuple consumed by `extraction_branching`.

In [11]:
def document_validator(file_path: str) -> tuple[str, bool, str]:
    """Reads a document and returns whether it is a document or an image

    Args:
        file_path (str): The path to the document or image file. In the context of the agent, this will be the path to the input that the user has uploaded. This function is expected to account for all the different file types that the user might upload. 
        
        Given that there can be multiple types of documents that can be uploaded, and a number of file uploads, only the following formats are expected to be uploaded:

        Raster formats:
        - "jpg", "jpeg", "jpe", "jif", "jfif", 
        - "png", 
        - "webp",  
        - "tif", "tiff", 
        - "heic", "heif", 
        - "bmp", 
        - "raw", "cr2", "nef", "arw"

        File formats:
        - "pdf", "txt"

    Returns: tuple[str, bool, str]
        str: The file path. Returns None if the file is incompatible
        bool: Whether the file is a compatible format for the agent to process
        str: The type of file that was uploaded (raster, document). Returns None when incompatible file format uploaded 
    """

    normalized_file_path = normalize_file_path(file_path)
    file_path = normalized_file_path[0]
    extension = normalized_file_path[1]

    # [CHANGED] Use logger instead of print
    # logger.info(f"Extension of {file_path} is {extension}")

    file_type = classify_file_type(extension)

    # [CHANGED] Return as tuple for consistency with docstring
    return (file_path, True, file_type)

## Phase 2 — Extraction Helpers

### `render_page_to_b64`

Renders a PyMuPDF page to a PNG pixmap at 2× scale (matrix `(2, 2)`) and returns it as a base64 string. The 2× scale improves OCR accuracy for the vision model on low-resolution scans.

In [12]:
def render_page_to_b64(page: fz.Page) -> str:
    """Renders a provided page as a b64 encoded string of bytes

    Args:
        page (fz.page): The page that needs to be encoded as an image

    Returns:
        str: b64 encoded byte string
    """
    page_matrix = fz.Matrix(2, 2)
    page_pix = page.get_pixmap(matrix=page_matrix)
    return image_encoder(page_pix.tobytes("png"))

### `extraction_branching`

Routes a validated file to the correct extraction strategy:

- **Raster files** — opened with Pillow, saved to a PNG byte buffer, encoded to base64, and returned as a `ProcessedImage`.
- **PDF/text documents** — iterated page-by-page with per-page classification:
  - Pages with ≥150 characters and at least one font → `"text"` extraction via `page.get_text()`
  - Pages that are image-dominant (image area > 95% of page), have <50 characters, or have more XObjects than fonts → `"vision"` extraction via `render_page_to_b64`
  - Ambiguous pages → fallback to `"vision"` with a warning log

All pages are aggregated into a `ProcessedDocument`.

In [13]:
def extraction_branching(validated_file_tuple: tuple):
    """Directs the control flow to the appropriate functions for extraction

    Args:
        validated_file_tuple (tuple): The output from document_validator().
    """

    # explicitly check if the file has been validated
    if not validated_file_tuple[1]:
        logger.critical(f"Skipping processing for invalid file: {validated_file_tuple[0]}")
        # print(f"Skipping processing for invalid file: {validated_file_tuple[0]}")  # Log intentional skip
        return
    
    if validated_file_tuple[1]:
        # the document is valid and can be further processed for extraction
        file_path = validated_file_tuple[0]
        file_type = validated_file_tuple[2]

        if file_type == "raster":
            # images need to be processed before sending into the model for extraction

            image_byte_arr = io.BytesIO()

            # encode the image and retrieve the base64 encoding
            try:
                image = Image.open(file_path)
                # save the image bytes to the byte array
                image.save(image_byte_arr, format='PNG')
                retrieved_image_bytes = image_byte_arr.getvalue()

                base64_image_encoding = image_encoder(retrieved_image_bytes)

                processed_raster_file = ProcessedImage(
                    file_path = file_path,
                    extraction_method = "vision",
                    image_b64_encoded = base64_image_encoding
                )
                
                return processed_raster_file

            except Exception as image_encoding_func_call_exp:
                # print(f"Exception occurred while calling the image_encoder() inside extraction_branching(): {image_encoding_func_call_exp.with_traceback()}\n")
                logger.error(f"Exception in image_encoder(): {image_encoding_func_call_exp}", exc_info=True)
                # after this, you need to send it over to the vision-first function

        elif file_type == "document":
            # documents need to be further processed for extraction
            # print(f"Document: {file_path}")
            logger.info(f"Document: {file_path}")
            document = fz.open(filename=Path(file_path))

            """How the document will be flagged as scanned or not:
                - get the total count of page fonts
                - get the total count of XObjects in the page
                - check the number of pages where images take up more area
                - check the number of drawings and tables

                conditional logic for classification:
                    -> if the number of fonts > XObjects:
                        if the number of pages where images take up more area is lesser than half:
                            * page gets flagged as text and can be used for text extraction
                    -> else:
                        if the number of pages where images take up more area is more than half the count of pages:
                            if the number of drawings and images > 0:
                                * page is flagged as scanned and needs to go for visual extraction
                
                We are performing page level classification and not on the entire document
            """

            # track the pages flagged for text extraction
            text_extraction_flagged_page_count = []
            processed_page_data_list: list[ProcessedPage] = []

            for page in document:
                # store the fonts, XObjects, and the text from the page
                page_fonts = set()
                page_xobjects = set()
                page_character_count = 0
                page_drawings, total_page_images = 0, 0

                page_character_count = len(page.get_text()) # get the number of characters in the page

                # retrieve the fonts and XObjects and add them to the respective sets
                for font in page.get_fonts():
                    page_fonts.add(font[0])
                for xobject in page.get_xobjects():
                    page_xobjects.add(xobject[0])
                

                # get the number of images and drawings
                page_images = page.get_images()
                total_page_images += len(page_images)
                page_drawings += len(page.get_drawings())

                # check the area occupied by images in a page compared to text
                page_area = abs(page.rect)
                is_page_image_dominant = False

                for image_tuple in page_images:
                    try:
                        image_rectangle = page.get_image_bbox(image_tuple)
                        if page_area > 0 and (abs(image_rectangle) / page_area) > 0.95:
                            is_page_image_dominant = True
                    except Exception:
                        continue
                
                if page_character_count > 150 and len(page_fonts) > 0:
                    text_extraction_flagged_page_count.append(True)

                    page_text = page.get_text()

                    processed_page_data = ProcessedPage(
                        page_number = page.number,
                        extraction_method = "text",
                        raw_content = page_text,
                        image_b64_encoded = None
                    )
                    processed_page_data_list.append(processed_page_data)

                elif page_character_count < 50 or is_page_image_dominant or len(page_xobjects) >= len(page_fonts):
                    text_extraction_flagged_page_count.append(False)

                    # # convert the current page into an image and pass it into the b64 image encoder function
                    # page_matrix = fz.Matrix(2, 2)
                    # page_pix = page.get_pixmap(matrix=page_matrix)
                    
                    # # retrieve the image bytes and pass it into the function
                    # page_image_bytes = page_pix.tobytes("png")
                    b64_encoded_page_image = render_page_to_b64(page)

                    processed_page_image_data = ProcessedPage(
                        page_number = page.number,
                        extraction_method = "vision",
                        raw_content = None,
                        image_b64_encoded = b64_encoded_page_image
                    )
                    processed_page_data_list.append(processed_page_image_data)

                else:
                    logger.warning(f"Page {page.number} is ambiguous (char_count={page_character_count}, font_count={len(page_fonts)}) - routing to vision")
                    text_extraction_flagged_page_count.append(False)
                    # # convert the current page into an image and pass it into the b64 image encoder function
                    # page_matrix = fz.Matrix(2, 2)
                    # page_pix = page.get_pixmap(matrix=page_matrix)
                    
                    # # retrieve the image bytes and pass it into the function
                    # page_image_bytes = page_pix.tobytes("png")
                    b64_encoded_page_image = render_page_to_b64(page)

                    processed_page_image_data = ProcessedPage(
                        page_number = page.number,
                        extraction_method = "vision",
                        raw_content = None,
                        image_b64_encoded = b64_encoded_page_image
                    )
                    processed_page_data_list.append(processed_page_image_data)

            # print(f"Number of pages flagged for text extraction: {len([page for page in text_extraction_flagged_page_count if page == True])}")
            # print(f"Number of pages flagged for vision extraction: {len([page for page in text_extraction_flagged_page_count if page == False])}")
            logger.info(f"Number of pages flagged for text extraction: {len([page for page in text_extraction_flagged_page_count if page == True])}")
            logger.info(f"Number of pages flagged for vision extraction: {len([page for page in text_extraction_flagged_page_count if page == False])}")

            processed_document =  ProcessedDocument(
                file_path = file_path,
                total_pages = document.page_count,
                pages = processed_page_data_list
            )
            return processed_document


    elif validated_file_tuple[1] == False:
        # print(f"Incompatible file uploaded.\n")
        logger.error(f"Incompatible file uploaded.\n")

### `image_encoder`

Thin wrapper around `base64.b64encode`. Raises `ImageEncodingException` on empty input or encoding failure, keeping error handling consistent with the rest of the pipeline.

In [14]:
def image_encoder(image_bytes: bytes) -> str:
    """Encode image bytes or scanned document page bytes to a base64 string.

    Args:
        image_bytes (bytes): Raw image bytes that need to be encoded.

    Returns:
        str: The resultant base64 string.
    """

    if not image_bytes:
        raise ImageEncodingException(
            message="Cannot encode empty image bytes"
        )

    try:
        return base64.b64encode(image_bytes).decode()
    except Exception as err:
        raise ImageEncodingException(
            message=f"Exception occurred while trying to encode the image bytes: {err}"
        ) from err

## Phase 3 — Output Validation

### `validate_clinical_output` and `attempt_json_recovery`

Gemma 4's output is raw text. These two functions clean and validate it into a usable dict:

1. **Strip markdown fences** — models often wrap JSON in ` ```json ``` ` blocks.
2. **Parse JSON** — if this fails, `attempt_json_recovery` tries to close unclosed brackets caused by `max_new_tokens` truncation.
3. **Key validation** — any missing required keys (`document_type`, `patient_context`, `lab_findings`, `imaging_findings`, `clinical_notes`, `flagged_signals`, `extraction_confidence`) are filled with `None` rather than crashing.
4. **Parse failure fallback** — returns a skeleton dict with `"extraction_confidence": "low"` and the raw response attached for inspection.

In [15]:
def validate_clinical_output(raw_response: str) -> dict:
    """Parses and validates the raw JSON response from Gemma 4.

    Args:
        raw_response (str): The raw string output from the model

    Returns:
        dict: Validated clinical profile, or a partial result with error flag
    """
    cleaned = raw_response.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("```")[1]
        if cleaned.startswith("json"):
            cleaned = cleaned[4:]
        cleaned = cleaned.strip()

    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError:
        # Attempt recovery — truncation leaves JSON unclosed
        # Try to salvage whatever parsed cleanly before the cut
        logger.warning("JSON parse failed — attempting truncation recovery")
        cleaned = attempt_json_recovery(cleaned)
        try:
            parsed = json.loads(cleaned)
        except json.JSONDecodeError as e:
            logger.error(f"Recovery failed: {e}")
            return {
                "document_type": "unknown",
                "patient_context": None,
                "lab_findings": [],
                "imaging_findings": [],
                "clinical_notes": [],
                "flagged_signals": [],
                "extraction_confidence": "low",
                "parse_error": str(e),
                "raw_response": raw_response
            }

    # validate keys...
    required_keys = {
        "document_type", "patient_context", "lab_findings",
        "imaging_findings", "clinical_notes",
        "flagged_signals", "extraction_confidence"
    }
    missing = required_keys - parsed.keys()
    for key in missing:
        parsed[key] = None
    return parsed


def attempt_json_recovery(broken_json: str) -> str:
    """Attempts to close a truncated JSON string by balancing brackets.
    
    Args:
        broken_json (str): The incomplete JSON string
        
    Returns:
        str: A potentially valid JSON string with brackets closed
    """
    # Count unclosed structures
    open_braces = broken_json.count('{') - broken_json.count('}')
    open_brackets = broken_json.count('[') - broken_json.count(']')

    # Strip trailing incomplete key-value pair — find last complete entry
    # Truncation often leaves a dangling comma or partial string
    last_complete = max(
        broken_json.rfind('}'),
        broken_json.rfind(']')
    )
    if last_complete != -1:
        broken_json = broken_json[:last_complete + 1]

    # Recount after trimming
    open_braces = broken_json.count('{') - broken_json.count('}')
    open_brackets = broken_json.count('[') - broken_json.count(']')

    # Close in reverse order
    broken_json += ']' * open_brackets
    broken_json += '}' * open_braces

    return broken_json

## Phase 4 — Clinical Signal Extraction

### `extract_clinical_signals`

Constructs the multimodal prompt for Gemma 4 from a processed document or image. Handles two cases:

- **`ProcessedDocument`** — iterates over pages, appending text blocks or base64 image blocks depending on each page's `extraction_method`. The system prompt is prepended as the first content part.
- **`ProcessedImage`** — wraps the single b64 image in the same content format.

The returned `content_parts` list is passed directly to `processor.apply_chat_template()` in the inference cell.

The system prompt instructs the model to extract five signal categories — lab results, imaging findings, patient demographics, clinical notes, and flagged abnormalities — and return them as a structured JSON object matching the schema defined inline.

In [16]:
def extract_clinical_signals(processed_document: ProcessedDocument | ProcessedImage) -> dict:
    """Passes the extracted data into the Gemma models and returns a dict containing clinical signals/indicators

    Args:
        processed_document (ProcessedDocument | ProcessedImage): _description_

    Returns:
        dict: A dict object (intended to be JSON) containing clinical signals
    """

                # this will hold the instructions for the model
    base_system_prompt = """You are a Medical Document Analyst specialising in extracting clinical signals from medical documents (medical reports, lab results, pathology documents, radiology notes). You understand that an ordinary individual who does not have knowledge of understanding or interpreting needs more than just guidance; they need to be able to understand what they are looking at, signals that might have been overlooked, and the long term implications of the report they hold. And for that, you need to first extract the clinical signals from the document(s), which is what you do. You help in identifying clinical signals such as elevated markers, abnormal findings, flagged terms, and extracting them from the document. These signals are required to build a clinical profile of the patient.

    Extract key clinical signals from this medical document focusing on:
        1. Lab test results (normal vs abnormal ranges)
        2. Medication dosages (conversion units if needed)
        3. Imaging findings (shape, location, contrast)
        4. Patient demographics (age/gender/chat)
        5. Diagnosis implications

    Extract the clinical signals carefully with accuracy and precision. Construct a structured clinical profile of the patient using the extracted data, and provide the clinical profile as a single JSON object, following the provided JSON schema below strictly:
    
    {
        "document_type": "lab_report | radiology | pathology | clinical_note | unknown",
        "patient_context": {
            "age": "number or null",
            "sex": "string or null",
            "stated_history": "string or null"
        },
        "lab_findings": [
            {
            "test_name": "string",
            "value": "number or string",
            "unit": "string or null",
            "reference_range": "string or null",
            "status": "normal | low | high | critical | unknown"
            }
        ],
        "imaging_findings": [
            {
            "modality": "X-ray | MRI | CT | Ultrasound | other",
            "region": "string",
            "observation": "string"
            }
        ],
        "clinical_notes": ["string"],
        "flagged_signals": [
            {
            "signal": "string",
            "reason": "string"
            }
        ],
        "extraction_confidence": "high | medium | low"
    }
    """ 

    # check whether the input is an image or document list and branch accordingly
    match processed_document:
        case ProcessedDocument():
            # here, each page needs to be looped over and parsed according to whether it needs an image extraction or text extraction
            # a single prompt is leveraged for this
            """prompt construction:
            * have the base system prompt that instructs the model on what it needs to perform
            * have a prompt holder -- a base str that will hold the entire prompt
            
            - if the page needs to a text extraction, extract text and add it to the prompt holder
            - if the page needs vision extraction, convert it to b64 encoding, and add it as an image with a placeholder [according to the gemma 4 prompt format]

            continue this step for all pages
            """

            content_parts = [] # this will hold the data from all the pages

            # attach the instructions for the model - the system prompt
            content_parts.append({
                "type": "text",
                "text": base_system_prompt
            })

            file_path = processed_document.file_path
            processed_document_pages = processed_document.pages

            # access each ProcessedPage object in the list
            for processed_page in processed_document_pages:
                # branch according to the extraction_method specified for the respective page
                match processed_page.extraction_method:
                    case "text":
                        content_parts.append({
                            "type": "text",
                            "text": processed_page.raw_content
                        })
                    case "vision":
                        content_parts.append({
                            "type": "image",
                            "image": processed_page.image_b64_encoded # handled by the processor
                        })

            return content_parts
            

        case ProcessedImage():
            # this here is just an image uploaded by the user and we assume that there is no additional context provided by them
            if processed_document.image_b64_encoded is not None:

                content_parts = [] # this will hold the content of the image(s)
                content_parts.append({ # attach the system prompt
                    "type": "text",
                    "text": base_system_prompt
                })

                # extract the b64 encodings
                b64_image_bytes = processed_document.image_b64_encoded
                # add it to the list of contents
                content_parts.append({
                    "type": "image",
                    "image": b64_image_bytes
                })

                return content_parts
            else:
                logger.error(f"ProcessedImageObject does not contain b64 encodings. Cannot proceed with vision extraction")
                return
    


        case _:
            return "unknown type"


## Batch Inference Entry Point

### `main`

Iterates over all files in the test data directory, running the full pipeline — validation → extraction branching → clinical signal extraction — and logging results. Intended for batch testing across a dataset rather than single-document inference.

In [17]:
def main():
    """The main function of the Document Parser agent
    """
    print("Document parser agent execution commenced...\n")

    # contains the testing documents
    testing_data_directory = Path("/kaggle/input/datasets/shwetahiyaa/dataset-3/PMC1/PMC1/PMC100")
    
    for file_path in testing_data_directory.iterdir():
        if file_path.is_file():

            # start by performing the document validation
            validation_tuple = document_validator(file_path)

            # pass the validation tuple to the extractor branching function
            result = extraction_branching(validation_tuple)
            # logger.info(f"Result: {result}")

            # call the clinical signals extraction function
            if result is not None:
                clinical_signals = extract_clinical_signals(result)
                logger.info(f"Clinical signals: {clinical_signals}")
            print("-------\n")

## Model Loading

Loads the Gemma 4 processor and model from Kaggle's model hub. Two model sizes are available:

- `gemma-4-e2b-it` — 2B parameter instruction-tuned variant; used here for faster iteration.
- `gemma-4-e4b-it` — 4B variant; higher accuracy, more VRAM required.

The model is loaded in `bfloat16` with `device_map="auto"` to distribute across available GPUs automatically.

In [18]:
GEMMA4_E4B_MODEL_ID = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1"
GEMMA4_E2B_MODEL_ID = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1"
# load the model
processor = AutoProcessor.from_pretrained(GEMMA4_E2B_MODEL_ID)
model = Gemma4ForConditionalGeneration.from_pretrained(
    GEMMA4_E2B_MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

### GPU Memory Flush

Clears the CUDA cache and triggers Python garbage collection before inference. Called once here after model load, and again inside the inference loop before each document to prevent OOM errors on long runs.

In [19]:
torch.cuda.empty_cache()
gc.collect()

35

## Multi-file Inference Loop

### Inference Function — `run_inference_for_clinical_signal_extraction`

Wraps the full pipeline into a single callable for batch inference over a directory.
This replaces the ad-hoc loop in the cell above and makes the pipeline reusable across
different datasets and model configurations without duplicating inference code.

**Pipeline steps per file:**
1. `document_validator` — checks format compatibility and classifies as `raster` or `document`
2. `extraction_branching` — extracts content into a `ProcessedDocument` or `ProcessedImage`
3. GPU cache flush — prevents OOM accumulation across files
4. `extract_clinical_signals` — builds the multimodal prompt payload
5. `processor.apply_chat_template` — tokenizes and formats for Gemma 4
6. `model.generate` — greedy decoding (`do_sample=False`) for deterministic JSON output
7. Token slicing — decodes only newly generated tokens by offsetting from `input_ids.shape[1]`
8. `validate_clinical_output` — parses, recovers if truncated, and fills any missing schema keys

**Returns:** `{file_path_str: clinical_profile_dict}` — one entry per successfully processed file.
Files that fail validation or produce a `None` extraction result are silently skipped.

> **Note on `BitsAndBytesConfig`:** The import at the top of this cell is unused here — 
> if you intend to run quantized inference (e.g. 4-bit NF4), pass a `BitsAndBytesConfig` 
> to `Gemma4ForConditionalGeneration.from_pretrained()` at model load time rather than here.

In [20]:
from transformers import BitsAndBytesConfig


def run_inference_for_clinical_signal_extraction(
    model, 
    processor,
    data_source_directory: str,
) -> dict:
    """Runs the full document parsing and clinical signal extraction pipeline over a directory of medical files.

    For each file in the directory, the function validates the file format, extracts content
    (text or vision depending on page classification), constructs a multimodal prompt, runs
    Gemma 4 inference, and validates the structured JSON output. GPU memory is flushed before
    each file to prevent OOM errors on longer runs.

    Args:
        model: A loaded Gemma4ForConditionalGeneration instance.
        processor: The corresponding AutoProcessor for the model.
        data_source_directory (str): Path to the directory containing medical documents or images to process.

    Returns:
        dict: A mapping of file path strings to validated clinical profile dicts. Each dict follows
        the schema defined in validate_clinical_output() — keys include document_type, patient_context,
        lab_findings, imaging_findings, clinical_notes, flagged_signals, and extraction_confidence.
        Files that fail validation or extraction are skipped and not included in the output.
    """

    # stores all the clinical signals
    all_results = {}

    directory_path = Path(data_source_directory)
    for file_path in directory_path.iterdir():
        if file_path.is_file():
    
            # start by performing the document validation
            validation_tuple = document_validator(file_path)
    
            # pass the validation tuple to the extractor branching function
            result = extraction_branching(validation_tuple)
            # logger.info(f"Result: {result}")
    
            # call the clinical signals extraction function
            if result is not None:
                # empty the gpu memory before scanning this document
                torch.cuda.empty_cache()
                gc.collect()
                
                prompt_payload = extract_clinical_signals(result)
                # logger.info(f"Clinical signals: {clinical_signals}")
    
                messages = [
                    {
                        "role": "user",
                        "content": prompt_payload
                    }
                ]
    
                # apply chat template
                inputs = processor.apply_chat_template(
                    messages,
                    add_generation_prompt=True,
                    tokenize=True,
                    return_dict=True,
                    return_tensors="pt"
                ).to(model.device)
    
                with torch.no_grad():
                    output_ids = model.generate(
                        **inputs,
                        max_new_tokens=2048,
                        do_sample=False, # keeps it deterministic without random sampling - needed for structured JSON output
                        temperature=1.0 # even though ignored when sampling is false, needed for some versions
                    )
    
                # decode only the newly generated tokens
                # Use .shape[1] to get the integer value of the sequence length
                raw_response = processor.decode(
                    output_ids[0][inputs["input_ids"].shape[1]:], 
                    skip_special_tokens=True
                )
    
                logger.info(f"Raw model response: {raw_response}")
                # print(validate_clinical_output(raw_response))

                clinical_signals = validate_clinical_output(raw_response)
                all_results[str(file_path)] = clinical_signals
            print("="*80, end="\n")

    return all_results

## Single-Document Inference Loop

Runs the full pipeline on the test dataset and prints validated clinical profiles. Steps per file:

1. Validate and classify the file with `document_validator`
2. Extract content into a `ProcessedDocument` or `ProcessedImage` via `extraction_branching`
3. Build the multimodal prompt via `extract_clinical_signals`
4. Apply the Gemma 4 chat template and tokenize
5. Generate with greedy decoding (`do_sample=False`) for deterministic JSON output
6. Decode only the newly generated tokens (slicing from `input_ids.shape[1]`)
7. Validate and print the parsed clinical profile

In [21]:
# Test out the model on a single directory

# testing_data_directory = Path("/kaggle/input/datasets/adrinorosario/vision-required-test-dataset")
# for file_path in testing_data_directory.iterdir():

file_path = Path("/kaggle/input/datasets/adrinorosario/vision-required-test-dataset/main.pdf")
if file_path.is_file():

    # start by performing the document validation
    validation_tuple = document_validator(file_path)

    # pass the validation tuple to the extractor branching function
    result = extraction_branching(validation_tuple)
    # logger.info(f"Result: {result}")

    # call the clinical signals extraction function
    if result is not None:
        # empty the gpu memory before scanning this document
        torch.cuda.empty_cache()
        gc.collect()
        
        prompt_payload = extract_clinical_signals(result)
        # logger.info(f"Clinical signals: {clinical_signals}")

        messages = [
            {
                "role": "user",
                "content": prompt_payload
            }
        ]

        # apply chat template
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=2048,
                do_sample=False, # keeps it deterministic without random sampling - needed for structured JSON output
                temperature=1.0 # even though ignored when sampling is false, needed for some versions
            )

        # decode only the newly generated tokens
        # Use .shape[1] to get the integer value of the sequence length
        raw_response = processor.decode(
            output_ids[0][inputs["input_ids"].shape[1]:], 
            skip_special_tokens=True
        )

        logger.info(f"Raw model response: {raw_response}")
        print(validate_clinical_output(raw_response))
    print("="*80, end="\n")

INFO:__main__:Extension of /kaggle/input/datasets/adrinorosario/vision-required-test-dataset/main.pdf is .pdf
INFO:__main__:Document: /kaggle/input/datasets/adrinorosario/vision-required-test-dataset/main.pdf
INFO:__main__:Number of pages flagged for text extraction: 6
INFO:__main__:Number of pages flagged for vision extraction: 0
INFO:__main__:Raw model response: ```json
{
    "document_type": "radiology",
    "patient_context": {
        "age": 71,
        "sex": "male",
        "stated_history": "Presented with intermittent headaches for the past 4 months (improved with analgesics), abrupt loss of vision in the past 4 months, and weakness of left extremities since 1 year ago, especially with left extremities. No seizures, anosmia, hearing loss, or vomiting. Good appetite and no weight loss. History of stroke 1 year ago. Routine lab and immunoserology were within normal limits, including negative HIV antibodies."
    },
    "lab_findings": [
        {
            "test_name": "Routin

{'document_type': 'radiology', 'patient_context': {'age': 71, 'sex': 'male', 'stated_history': 'Presented with intermittent headaches for the past 4 months (improved with analgesics), abrupt loss of vision in the past 4 months, and weakness of left extremities since 1 year ago, especially with left extremities. No seizures, anosmia, hearing loss, or vomiting. Good appetite and no weight loss. History of stroke 1 year ago. Routine lab and immunoserology were within normal limits, including negative HIV antibodies.'}, 'lab_findings': [{'test_name': 'Routine laboratory and immunoserology examinations', 'value': 'Within normal limits (except for MRS metabolites)', 'unit': None, 'reference_range': 'Normal limits', 'status': 'normal'}], 'imaging_findings': [{'modality': 'CT scan', 'region': 'Right subcortical parietal lobe, left thalamus, left basal ganglia, right cortical-subcortical occipital lobe', 'observation': 'Multiple calcified nodules found. Calcified nodule with diameter of 1.20 cm